In [1]:
import os, sys, shutil, subprocess
from multiprocessing import cpu_count

print("Python:", sys.version)
print("CPU cores:", cpu_count())

# Kaggle paden
BASE = "/kaggle/working"
APPLIO_DIR = f"{BASE}/Applio"
LOGS_DIR = f"{APPLIO_DIR}/logs"

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
CPU cores: 4


In [2]:
APPLIO_REF = "3.6.2"   # probeer bv "main" of een release-tag als dat beter werkt

%cd /kaggle/working
if not os.path.isdir(APPLIO_DIR):
    !git clone https://github.com/IAHispano/Applio --branch {APPLIO_REF} --single-branch

# System deps (portaudio is vaak nodig)
!apt-get update -y
!apt-get install -y portaudio19-dev psmisc

%cd {APPLIO_DIR}

# Snelle/strakke pip via uv (zoals Applio notebooks doen)
!pip -q install uv
!uv pip install -q -r requirements.txt --extra-index-url https://download.pytorch.org/whl/cu128 --index-strategy unsafe-best-match --system

# Download Applio prerequisites (models/pretraineds etc.)
!python core.py prerequisites --models "True" --pretraineds_hifigan "True" > /dev/null 2>&1

print("✅ Install klaar")

/kaggle/working
Cloning into 'Applio'...
remote: Enumerating objects: 19967, done.
remote: Counting objects: 100% (121/121), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 19967 (delta 84), reused 72 (delta 60), pack-reused 19846 (from 3)
Receiving objects: 100% (19967/19967), 49.28 MiB | 19.28 MiB/s, done.
Resolving deltas: 100% (13064/13064), done.
Note: switching to 'c7be7282af0f7b6382cbb2ac780b11242da12916'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Get:1 https://cli.github.com/packages stable 

In [3]:
!python /kaggle/working/Applio/core.py preprocess \
  --model_name "hitsugi_rvc_v2" \
  --dataset_path "/kaggle/input/datasets/michaeltje26/hitsugi" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --cut_preprocess "Automatic" \
  --process_effects "False" \
  --noise_reduction "False" \
  --chunk_len "3" \
  --overlap_len "0.3" \
  --normalization_mode "none"

Starting preprocess with 4 processes...
100%|█████████████████████████████████████████████| 4/4 [00:32<00:00,  8.19s/it]
Preprocess completed in 32.78 seconds on 00:34:24 seconds of audio.


In [4]:
!python /kaggle/working/Applio/core.py extract \
  --model_name "hitsugi_rvc_v2" \
  --f0_method "rmvpe" \
  --sample_rate "40000" \
  --cpu_cores "4" \
  --gpu "0" \
  --embedder_model "contentvec" \
  --embedder_model_custom "" \
  --include_mutes "2"

Starting pitch extraction on cuda:0 using rmvpe...
100%|█████████████████████████████████████████| 645/645 [00:17<00:00, 37.52it/s]
Pitch extraction completed in 24.97 seconds.
Starting embedding extraction with 4 cores on cuda:0...
100%|█████████████████████████████████████████| 645/645 [00:10<00:00, 60.21it/s]
Embedding extraction completed in 17.55 seconds.


In [5]:
!python /kaggle/working/Applio/core.py index \
  --model_name "hitsugi_rvc_v2" \
  --index_algorithm "Auto"

Saved index file '/kaggle/working/Applio/logs/hitsugi_rvc_v2/hitsugi_rvc_v2.index'


In [6]:
!python /kaggle/working/Applio/core.py train \
  --model_name "hitsugi_rvc_v2" \
  --save_every_epoch "20" \
  --save_only_latest "True" \
  --save_every_weights "False" \
  --total_epoch "200" \
  --sample_rate "40000" \
  --batch_size "10" \
  --gpu 0 \
  --pretrained "True" \
  --custom_pretrained "False" \
  --overtraining_detector "False" \
  --cleanup "False" \
  --cache_data_in_gpu "False" \
  --vocoder "HiFi-GAN" \
  --checkpointing "False"

2026-03-04 17:39:18.118076: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772645958.337117    1159 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772645958.409794    1159 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772645958.962097    1159 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772645958.962157    1159 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772645958.962163    1159 computation_placer.cc:177] computation placer alr

In [7]:
import os

paths = [
    "/kaggle/working/hitsugi_rvc_v2_inference.zip",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/wavs",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/raw",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/f0",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/features",
    "/kaggle/working/Applio/logs/hitsugi_rvc_v2/tensorboard"
]

for p in paths:
    if os.path.exists(p):
        if os.path.isdir(p):
            import shutil
            shutil.rmtree(p)
        else:
            os.remove(p)
        print("Deleted:", p)


Deleted: /kaggle/working/Applio/logs/hitsugi_rvc_v2/f0


In [ ]:
import torch, json
from pathlib import Path
from collections import OrderedDict

RUN_DIR = Path("/kaggle/working/Applio/logs/hitsugi_rvc_v2")

# 1) Pak nieuwste generator checkpoint (G_*.pth)
g_list = sorted(RUN_DIR.glob("G_*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
if not g_list:
    raise FileNotFoundError(f"Geen G_*.pth gevonden in {RUN_DIR}")
G_PATH = g_list[0]

ckpt = torch.load(G_PATH, map_location="cpu")

# 2) Pak weights (kan per checkpoint net anders heten)
if isinstance(ckpt, dict) and "model" in ckpt:
    weight = ckpt["model"]
elif isinstance(ckpt, dict) and "weight" in ckpt:
    weight = ckpt["weight"]
else:
    # soms is ckpt zelf al een state_dict
    weight = ckpt

# 3) Sample rate bepalen (uit config.json als die er is, anders fallback 40k)
sr_int = 40000
cfg_path = RUN_DIR / "config.json"
if cfg_path.exists():
    cfg_json = json.loads(cfg_path.read_text())
    sr_int = int(cfg_json.get("sr", sr_int))

# 4) Maak een "Marilyn-achtige" RVC v2 config (18 items)
#    (dit is het format dat jouw Applio error verwachtte)
def rvc_v2_config_18(sr: int):
    # upsample settings per sample rate (standaard RVC)
    if sr == 48000:
        upsample_rates = [12, 10, 2, 2]
        upsample_kernel_sizes = [24, 20, 4, 4]
    elif sr == 40000:
        upsample_rates = [10, 10, 2, 2]
        upsample_kernel_sizes = [20, 20, 4, 4]
    elif sr == 32000:
        upsample_rates = [8, 8, 2, 2]
        upsample_kernel_sizes = [16, 16, 4, 4]
    else:
        # als iemand een gekke sr heeft: probeer dichtstbijzijnde
        upsample_rates = [10, 10, 2, 2]
        upsample_kernel_sizes = [20, 20, 4, 4]

    # dit is dezelfde “stijl” als het werkende Marilyn model
    return [
        1025,                 # n_fft
        32,                   # hop_length
        192,                  # inter_channels
        192,                  # hidden_channels
        768,                  # filter_channels
        2,                    # n_heads
        6,                    # n_layers
        3,                    # kernel_size
        0,                    # p_dropout
        "1",                  # resblock
        [3, 7, 11],           # resblock_kernel_sizes
        [[1, 3, 5], [1, 3, 5], [1, 3, 5]],  # resblock_dilation_sizes
        upsample_rates,       # upsample_rates
        512,                  # upsample_initial_channel
        upsample_kernel_sizes,# upsample_kernel_sizes
        109,                  # spk_embed_dim
        256,                  # gin_channels
        sr,                   # sr (int) als laatste item
    ]

config18 = rvc_v2_config_18(sr_int)

# sr key (zoals Marilyn: "48k", dus bij jou "40k")
sr_key = f"{sr_int//1000}k"

export = OrderedDict({
    "weight": weight,
    "config": config18,
    "info": "export_like_marilyn",
    "sr": sr_key,
    "f0": 1,
    "version": "v2",   # jij zei: v2 (prima als label)
})

OUT = RUN_DIR / "hitsugi_rvc_v2_export.pth"
torch.save(export, OUT)

print("Exported:", OUT)
print("config len:", len(export["config"]))
print("sr:", export["sr"], "config[-1]:", export["config"][-1])

In [1]:
import torch, json
from pathlib import Path

RUN_DIR = Path("/kaggle/working/Applio/logs/hitsugi_rvc_v2")

# pak nieuwste G
g_list = sorted(RUN_DIR.glob("G_*.pth"), key=lambda p: p.stat().st_mtime, reverse=True)
G_PATH = g_list[0]

cfg = json.loads((RUN_DIR / "config.json").read_text())

export = {
    "weight": torch.load(G_PATH, map_location="cpu")["model"],
    "config": [40, 256, 32, 8192, 2, 2, 2, 2, 2, 2, 512, 256],  # RVC v2.5 architectuur
    "sr": cfg.get("sr", 40000),
    "f0": cfg.get("f0", 1),
    "version": "v2.5",
    "spk": cfg.get("spk", "hitsugi")
}

OUT = RUN_DIR / "hitsugi_rvc_v2_export.pth"
torch.save(export, OUT)

print("Exported:", OUT)




IndexError: list index out of range

In [9]:
%%bash
cd /kaggle/working/Applio/logs/hitsugi_rvc_v2

# combineer pth achter index
cat hitsugi_rvc_v2.index hitsugi_rvc_v2_export.pth > combined.index

ls -lh combined.index

-rw-r--r-- 1 root root 334M Mar  4 20:09 combined.index
